In [1]:
import numpy as np, pandas as pd, pickle, os
import matplotlib.pyplot as plt, seaborn as sns
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Conv1D,
GlobalMaxPooling1D, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
print("Ready!")

Ready!


In [2]:
df = pd.read_csv('cleaned_data.csv')
MAX_WORDS = 10000; MAX_LEN = 300
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'].astype(str))
sequences = tokenizer.texts_to_sequences(df['clean_text'].astype(str))
X_pad = pad_sequences(sequences, maxlen=MAX_LEN, truncating='post')
X_train, X_test, y_train, y_test = train_test_split(
X_pad, df['label'].values, test_size=0.2,
random_state=42, stratify=df['label'])
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (35696, 300), Test: (8925, 300)


In [3]:
model = Sequential([
Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
Conv1D(128, 5, activation='relu'),
GlobalMaxPooling1D(),
Dropout(0.3),
Dense(64, activation='relu'),
Dropout(0.3),
Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam',
loss='binary_crossentropy',
metrics=['accuracy'])
model.summary()

c:\Users\LOQ\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
es = EarlyStopping(monitor='val_loss', patience=3,
restore_best_weights=True)
history = model.fit(X_train, y_train,
epochs=10, batch_size=64,
validation_split=0.1, callbacks=[es], verbose=1)
print("CNN training complete!")


Epoch 1/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9674 - loss: 0.0854 - val_accuracy: 0.9978 - val_loss: 0.0093
Epoch 2/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9975 - loss: 0.0091 - val_accuracy: 0.9986 - val_loss: 0.0068
Epoch 3/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.9983 - val_loss: 0.0070
Epoch 4/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9998 - loss: 0.0011 - val_accuracy: 0.9986 - val_loss: 0.0074
Epoch 5/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9999 - loss: 4.4547e-04 - val_accuracy: 0.9989 - val_loss: 0.0079
CNN training complete!


In [5]:
y_prob = model.predict(X_test).flatten()
y_pred = (y_prob >= 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)
print(f"CNN Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(classification_report(y_test, y_pred,
target_names=['Fake','Real']))
os.makedirs('saved_models', exist_ok=True)
model.save('saved_models/cnn_model.h5')
pickle.dump(tokenizer,
open('saved_models/cnn_tokenizer.pkl','wb'))
print("CNN model saved!")

279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
CNN Accuracy: 0.9978 (99.78%)


              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4641
        Real       1.00      1.00      1.00      4284

    accuracy                           1.00      8925
   macro avg       1.00      1.00      1.00      8925
weighted avg       1.00      1.00      1.00      8925

CNN model saved!
